# MedDec ELECTRA — Training Notebook (Colab)

**Goal**: Fine-tune `google/electra-base-discriminator` on the MedDec MIMIC-III dataset to detect and classify medical decision spans.

**This notebook assumes you have already:**
- Uploaded your data to Google Drive (`02 Data/meddec-mimic-iii/`)
- Uploaded your code folder to Google Drive
- Completed Phase 1 (raw_text/, splits/ exist)

**Colab runtime**: Runtime → Change runtime type → **T4 GPU** (free tier is sufficient)

---

### About MLflow on Colab

MLflow logging (params, metrics, artifacts) works perfectly on Colab — every `mlflow.log_*` call writes to a local `mlruns/` folder.  
The **MLflow UI** (`mlflow ui`) is a web server that you cannot run in Colab's browser directly.  
**Workaround used here**: set the tracking URI to Google Drive so runs persist across sessions, then display results inline with `mlflow.search_runs()` + matplotlib.

You can also view the full UI later by downloading `mlruns/` and running `mlflow ui` locally.

## Step 0 — Mount Drive, install dependencies, copy code

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

In [ ]:
# Install required packages (transformers pinned to 4.39.3 for compatibility)
!pip install -q transformers==4.39.3 mlflow tqdm

In [ ]:
import shutil, sys
from pathlib import Path

# ── Edit these two paths to match your Drive layout ──────────────────────────
DRIVE_CODE_DIR = Path("/content/drive/MyDrive/AI4H-project-rework/04 Code/04 Code/med-decision-extraction")
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/AI4H-project-rework/02 Data")
# ─────────────────────────────────────────────────────────────────────────────

# Copy .py files to fast local storage (/content) — running directly from
# Drive is significantly slower due to per-file I/O latency.
LOCAL_CODE = Path("/content/meddec")
if LOCAL_CODE.exists():
    shutil.rmtree(LOCAL_CODE)
shutil.copytree(DRIVE_CODE_DIR, LOCAL_CODE)
sys.path.insert(0, str(LOCAL_CODE))

print("Code at :", LOCAL_CODE)
print("Data at :", DRIVE_DATA_DIR)
print("Files   :", [f.name for f in LOCAL_CODE.glob("*.py")])

## Step 1 — Verify GPU + paths

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if device.type == "cuda":
    print("GPU   :", torch.cuda.get_device_name(0))
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM  : {total_vram:.1f} GB")
else:
    print("WARNING: No GPU found — training will be very slow on CPU.")
    print("Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
MEDDEC_DIR = DRIVE_DATA_DIR / "meddec-mimic-iii"
SPLITS_DIR = MEDDEC_DIR / "splits"

checks = {
    "meddec-mimic-iii/data/"    : (MEDDEC_DIR / "data").exists(),
    "meddec-mimic-iii/raw_text/": (MEDDEC_DIR / "raw_text").exists(),
    "splits/train.txt"          : (SPLITS_DIR / "train.txt").exists(),
    "splits/val.txt"            : (SPLITS_DIR / "val.txt").exists(),
    "splits/test.txt"           : (SPLITS_DIR / "test.txt").exists(),
}

all_ok = True
for name, ok in checks.items():
    status = "OK" if ok else "MISSING"
    print(f"  [{status:7s}] {name}")
    if not ok:
        all_ok = False

if not all_ok:
    raise RuntimeError("Some required paths are missing — re-run Phase 1 scripts before training.")

n_json = len(list((MEDDEC_DIR / "data").glob("*.json")))
n_txt  = len(list((MEDDEC_DIR / "raw_text").glob("*.txt")))
print(f"\nJSON annotations : {n_json}")
print(f"Raw text files   : {n_txt}")

## Step 2 — Configure MLflow

We point MLflow at a folder inside Google Drive so experiment runs survive Colab session resets.  
The UI is not accessible in-browser here, but Step 6 displays results inline as a DataFrame + plots.

In [ ]:
import mlflow

MLFLOW_DIR = DRIVE_DATA_DIR / "mlruns"      # persisted to Drive
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(f"file://{MLFLOW_DIR}")

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Existing experiments:", [e.name for e in mlflow.search_experiments()])

## Step 3 — Training

Calls `train()` from `train.py`. Typical runtime on a free-tier T4:
- Dataset loading: ~30–60 s
- Per epoch (~81 batches): ~1–2 min
- **5 epochs total: ~10–15 min**

Checkpoints are saved to `/content/checkpoints/` (fast SSD).  
Step 7 copies `best_model.pt` back to Drive after training.

In [ ]:
from train import train

OUTPUT_DIR = Path("/content/checkpoints")

model = train(
    meddec_dir   = MEDDEC_DIR,
    splits_dir   = SPLITS_DIR,
    output_dir   = OUTPUT_DIR,
    model_name   = "google/electra-base-discriminator",
    num_epochs   = 5,
    batch_size   = 4,
    lr           = 4e-5,
    grad_accum   = 2,
    warmup_ratio = 0.1,
    max_len      = 512,
    experiment   = "meddec-electra",
)

## Step 4 — View training results

`mlflow.search_runs()` returns a DataFrame — one row per run, one column per logged metric/param.  
We then pull per-epoch values via `MlflowClient.get_metric_history()` to draw loss curves.

In [ ]:
import mlflow
import pandas as pd

runs = mlflow.search_runs(
    experiment_names=["meddec-electra"],
    order_by=["start_time DESC"],
)

# Pretty-print the most recent run's hyperparams + final metrics
latest   = runs.iloc[0]
run_id   = latest["run_id"]

params  = {k.replace("params.", ""): v for k, v in latest.items() if k.startswith("params.") and pd.notna(v)}
metrics = {k.replace("metrics.", ""): round(float(v), 4) for k, v in latest.items() if k.startswith("metrics.") and pd.notna(v)}

print("=== Latest Run ===")
print(f"  Run ID : {run_id}")
print(f"  Status : {latest['status']}")
print()
print("Hyperparameters:")
for k, v in params.items():
    print(f"  {k:20s} = {v}")
print()
print("Final metrics:")
for k, v in metrics.items():
    print(f"  {k:25s} = {v}")

In [ ]:
import matplotlib.pyplot as plt

client = mlflow.tracking.MlflowClient()

def get_history(metric_name):
    """Return list of (step, value) for a metric in the latest run."""
    return [(m.step, m.value) for m in client.get_metric_history(run_id, metric_name)]

train_loss_hist = get_history("epoch_train_loss")
val_loss_hist   = get_history("val_loss")
val_acc_hist    = get_history("val_token_acc")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Loss plot
if train_loss_hist:
    epochs, vals = zip(*train_loss_hist)
    ax1.plot(epochs, vals, label="Train loss", marker="o")
if val_loss_hist:
    epochs, vals = zip(*val_loss_hist)
    ax1.plot(epochs, vals, label="Val loss", marker="s")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Cross-Entropy Loss")
ax1.set_title("Training vs Validation Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Token accuracy plot
if val_acc_hist:
    epochs, vals = zip(*val_acc_hist)
    ax2.plot(epochs, [v * 100 for v in vals], label="Val token accuracy", marker="o", color="seagreen")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Validation Token Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle(f"Run: {run_id[:8]}...", fontsize=10, color="grey")
plt.tight_layout()
plt.show()

# Warn if model may not have converged
if val_loss_hist and val_loss_hist[-1][1] >= val_loss_hist[0][1]:
    print("WARNING: val loss did not decrease — model may not have converged. Try more epochs.")
else:
    print("Val loss decreased across training. Model converged normally.")

## Step 5 — Save checkpoint to Drive

The training run saved checkpoints to `/content/checkpoints/` (fast local SSD).  
This step copies them back to Drive so they survive session expiry.

In [ ]:
import shutil

DRIVE_CKPT_DIR = DRIVE_DATA_DIR / "checkpoints"
DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)

for fname in ["best_model.pt", "last_model.pt"]:
    src = OUTPUT_DIR / fname
    dst = DRIVE_CKPT_DIR / fname
    if src.exists():
        shutil.copy(src, dst)
        size_mb = src.stat().st_size / 1e6
        print(f"Saved {fname} ({size_mb:.0f} MB) → {dst}")
    else:
        print(f"WARNING: {fname} not found in {OUTPUT_DIR}")

## Step 6 — Sanity check: load checkpoint + run one forward pass

Verifies the saved model loads cleanly and produces output of the expected shape.  
Prints token-level predictions for the first 20 tokens of the first validation note.

In [ ]:
import torch
from model import MedDecModel
from dataset import load_electra_tokenizer, make_dataloader, NUM_LABELS

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = load_electra_tokenizer()

# Load the best checkpoint
model_ckpt = MedDecModel(num_labels=NUM_LABELS).to(device)
model_ckpt.load_state_dict(torch.load(OUTPUT_DIR / "best_model.pt", map_location=device))
model_ckpt.eval()
print("Checkpoint loaded.")

# Run one batch from the validation set
val_loader = make_dataloader(
    SPLITS_DIR / "val.txt", MEDDEC_DIR, tokenizer,
    train=False, batch_size=1, max_len=512,
)
batch  = next(iter(val_loader))
ids    = batch["input_ids"][:, :512].to(device)
mask   = batch["attention_mask"][:, :512].to(device)
labels = batch["labels"][:, :512]

with torch.no_grad():
    logits = model_ckpt(ids, mask)

preds  = logits.argmax(-1)[0].cpu().tolist()
gold   = labels[0].tolist()

print(f"Input  : {ids.shape}")
print(f"Logits : {logits.shape}   (batch=1, seq_len, 19 label classes)")
print()

# Decode first 20 tokens with predicted + gold labels
LABEL_NAMES = (
    {i * 2:     f"B-Cat{i+1}" for i in range(9)}
    | {i * 2 + 1: f"I-Cat{i+1}" for i in range(9)}
    | {18: "O", -100: "PAD"}
)

print(f"{'Tok':>4}  {'Token':20s}  {'Gold':12s}  {'Pred':12s}")
print("-" * 55)
for idx in range(min(20, ids.shape[1])):
    tok_str   = tokenizer.decode([ids[0, idx].item()])
    gold_str  = LABEL_NAMES.get(gold[idx],  str(gold[idx]))
    pred_str  = LABEL_NAMES.get(preds[idx], str(preds[idx]))
    match     = "" if gold[idx] == -100 else ("✓" if gold[idx] == preds[idx] else "✗")
    print(f"{idx:>4}  {tok_str:20s}  {gold_str:12s}  {pred_str:12s}  {match}")

---

## Next Steps

Training is complete. The token-level accuracy above is a **proxy metric** — it counts every O token as a correct prediction, which inflates the number.  

The real evaluation metric is **span-level F1**: a prediction is a true positive only if the exact (sample, category, start_token, end_token) tuple matches a gold annotation.

That is implemented in **Phase 2.4 — `evaluate.py`**:
- BIO decoding: scan predictions for `B-n` → `I-n` → `I-n` ... chains → extract (start, end, category) spans
- Compare predicted spans to gold spans
- Report precision, recall, F1 per category + macro F1

**Target**: span F1 ≈ 0.3–0.5 on this dataset size (full dataset + full training → 0.782 per Elgaar et al., 2024)